# Euro500 Equity Portfolio

This notebook constructs a quarterly rebalanced equity portfolio consisting of the 500 largest non-financial firms headquartered in euro-area countries. The sample starts in 1999Q1 and is used as the basis for subsequent return and event-study analyses.

## 0. Setup

In [21]:
# --- Imports & configuration ---
from pathlib import Path
import pandas as pd
import numpy as np
import lseg.data as ld
import time
import warnings

# --- Output paths (anpassen) ---
BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
(DATA_DIR := BASE_DIR / "intermediate").mkdir(parents=True, exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

warnings.filterwarnings(
    "ignore",
    message=r".*Downcasting behavior in `replace` is deprecated.*",
    category=FutureWarning,
    module=r"lseg\.data\._tools\._dataframe"
)

## 1. Definition of the Investment Universe

### 1.1 Euro-Area Headquartered Firms

The initial universe consists of all publicly listed, active, primary equity instruments whose headquarters are located in a euro-area country.

In [22]:
import pandas as pd

EURO_ADOPTION = {
    # Founding members (book money from 1999-01-01)
    "AT": "1999-01-01",
    "BE": "1999-01-01",
    "FI": "1999-01-01",
    "FR": "1999-01-01",
    "DE": "1999-01-01",
    "IE": "1999-01-01",
    "IT": "1999-01-01",
    "LU": "1999-01-01",
    "NL": "1999-01-01",
    "PT": "1999-01-01",
    "ES": "1999-01-01",

    # Later entrants
    "GR": "2001-01-01",
    "SI": "2007-01-01",
    "CY": "2008-01-01",
    "MT": "2008-01-01",
    "SK": "2009-01-01",
    "EE": "2011-01-01",
    "LV": "2014-01-01",
    "LT": "2015-01-01",
    "HR": "2023-01-01",
}

EURO_ADOPTION_DT = {k: pd.Timestamp(v) for k, v in EURO_ADOPTION.items()}

def euro_hq_codes_for_quarter(formation_date: pd.Timestamp) -> list[str]:
    """
    Countries eligible for the euro area in the return period
    following the portfolio formation date.
    """
    eligibility_date = formation_date + pd.Timedelta(days=1)
    return sorted(
        c for c, d in EURO_ADOPTION_DT.items()
        if d <= eligibility_date
    )

### Screen Builder

In [23]:
def make_screen_euro_all_for_quarter(formation_date: pd.Timestamp) -> str:
    """
    SCREEN universe for euro-area headquartered ordinary shares only.
    """

    codes = euro_hq_codes_for_quarter(formation_date)
    codes = [c for c in codes if isinstance(c, str) and c.strip() != ""]

    if not codes:
        codes_str = '"ZZ"'
    else:
        codes_str = ",".join(f'"{c}"' for c in codes)

    screen = (
        "SCREEN("
        # Only active, public, primary listings
        "U(IN(Equity(active,public,primary))),"
        # Only ordinary shares
        "TR.InstrumentType='Ordinary Shares',"
        # HQ filter
        f"IN(TR.HQCountryCode,{codes_str}),"
        # Normalize currency
        "CURN=EUR"
        ")"
    )

    return screen

In [24]:
def pull_euro_equities_snapshot(formation_date: pd.Timestamp, max_retries: int = 3, sleep_s: float = 2.0):
    formation_date = pd.Timestamp(formation_date).normalize()
    date_iso = formation_date.strftime("%Y-%m-%d")

    # SCREEN universe (string)
    universe = [make_screen_euro_all_for_quarter(formation_date)]

    fields = [
        "TR.RIC",
        "TR.PrimaryRIC",
        "TR.ISIN",
        "TR.SEDOL",
        "TR.CommonName",
        "TR.HeadquartersCountry",
        "TR.HQCountryCode",
        "TR.TRBCEconomicSector",
        "TR.CompanyMarketCap",
        "TR.MarketCap",
        "TR.FreeFloat",
        "TR.FreeFloatPct",
        "TR.Volume",
        "TR.PriceClose",
        "TR.InstrumentType",
    ]

    parameters = {
        "CURN": "EUR",
        "RH": "In",
        "CH": "Fd",
        "SDate": date_iso,
        "EDate": date_iso,
    }

    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            ld.open_session()
            try:
                df = ld.get_data(universe=universe, fields=fields, parameters=parameters)
            finally:
                ld.close_session()

            if df is None or df.empty:
                return pd.DataFrame()

            df = df.copy()

            # ---- IMPORTANT: strip empty identifiers (prevents downstream & some backend issues) ----
            for c in ["TR.RIC", "TR.PrimaryRIC", "TR.ISIN", "TR.SEDOL"]:
                if c in df.columns:
                    df[c] = df[c].astype(str).str.strip()
                    df.loc[df[c] == "", c] = pd.NA

            # IDs
            if "TR.RIC" in df.columns:
                df["RIC_snapshot"] = df["TR.RIC"]
            else:
                df["RIC_snapshot"] = pd.NA

            if "TR.PrimaryRIC" in df.columns:
                df["RIC_current"] = df["TR.PrimaryRIC"].fillna(df.get("TR.RIC"))
            else:
                df["RIC_current"] = df.get("TR.RIC")

            # MasterID: ISIN -> SEDOL -> RIC_current
            df["MasterID"] = df.get("TR.ISIN").fillna(df.get("TR.SEDOL")).fillna(df.get("RIC_current"))

            # Rename columns
            df = df.rename(columns={
                "TR.CommonName": "CompanyName",
                "TR.HeadquartersCountry": "HQCountry",
                "TR.HQCountryCode": "HQCountryCode",
                "TR.TRBCEconomicSector": "Sector",
                "TR.CompanyMarketCap": "CompanyMarketCap_EUR",
                "TR.MarketCap": "MarketCap_EUR",
                "TR.ISIN": "ISIN",
                "TR.SEDOL": "SEDOL",
                "TR.FreeFloat": "FreeFloat",
                "TR.FreeFloatPct": "FreeFloatPct",
                "TR.Volume": "Volume",
                "TR.PriceClose": "PriceClose",
                "TR.InstrumentType": "InstrumentType",
            })

            # Numeric coercion
            for c in ["CompanyMarketCap_EUR", "MarketCap_EUR", "FreeFloat", "FreeFloatPct", "Volume", "PriceClose"]:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors="coerce")

            return df

        except Exception as e:
            last_err = e
            time.sleep(sleep_s * attempt)

    raise last_err

In [25]:
for d in [
    pd.Timestamp("1998-12-31"),
    pd.Timestamp("2001-03-31"),
    pd.Timestamp("2008-03-31"),
    pd.Timestamp("2015-03-31"),
    pd.Timestamp("2025-03-31"),
]:
    print(d.date(), euro_hq_codes_for_quarter(d))

1998-12-31 ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'LU', 'NL', 'PT']
2001-03-31 ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'GR', 'IE', 'IT', 'LU', 'NL', 'PT']
2008-03-31 ['AT', 'BE', 'CY', 'DE', 'ES', 'FI', 'FR', 'GR', 'IE', 'IT', 'LU', 'MT', 'NL', 'PT', 'SI']
2015-03-31 ['AT', 'BE', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GR', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PT', 'SI', 'SK']
2025-03-31 ['AT', 'BE', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PT', 'SI', 'SK']


### 1.2 Period

Every Quater from 1999 to 2025

In [26]:
# Quarter ends: formation dates
START = "1998-12-31"   # formation for 1999Q1
END   = "2025-12-31"   # or pd.Timestamp.today().normalize()

q_ends = pd.date_range(START, END, freq="Q")

print("First quarters:", q_ends[:5])
print("Last quarters:", q_ends[-5:])

First quarters: DatetimeIndex(['1998-12-31', '1999-03-31', '1999-06-30', '1999-09-30',
               '1999-12-31'],
              dtype='datetime64[ns]', freq='QE-DEC')
Last quarters: DatetimeIndex(['2024-12-31', '2025-03-31', '2025-06-30', '2025-09-30',
               '2025-12-31'],
              dtype='datetime64[ns]', freq='QE-DEC')


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_76297/2188436847.py:5:FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.


## 2. Workspace Screener

In [27]:
# ------------------------------------------------------------
# Cache: quarterly snapshots
# ------------------------------------------------------------
CACHE_DIR = DATA_DIR / "euro_snap_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_path(formation_date: pd.Timestamp) -> Path:
    d = pd.Timestamp(formation_date).strftime("%Y-%m-%d")
    return CACHE_DIR / f"euro_snapshot_{d}.parquet"


# ------------------------------------------------------------
# Helpers: clean universes + robust retry wrapper
# ------------------------------------------------------------
def _clean_universe_list(x):
    """Drop NA/empty/whitespace-only and ensure unique strings."""
    s = pd.Series(x).dropna().astype(str)
    s = s[s.str.strip() != ""]
    return s.unique().tolist()

def _safe_get_data(universe, fields, parameters, max_retries=3, base_sleep=2.0):
    """
    Robust ld.get_data with:
    - hard-cleaned universe (removes '' which causes 'non-empty is required')
    - retry with backoff
    Returns DataFrame or raises last exception.
    """
    universe = _clean_universe_list(universe)
    if not universe:
        # Return empty df (prevents hard fail loops)
        return pd.DataFrame()

    last = None
    for r in range(1, max_retries + 1):
        try:
            df = ld.get_data(universe=universe, fields=fields, parameters=parameters)
            if df is None:
                return pd.DataFrame()
            return df
        except Exception as e:
            last = e
            time.sleep(base_sleep * r)
    raise last


# ------------------------------------------------------------
# Main: pull quarterly Euro equities snapshot (safe + cached)
# ------------------------------------------------------------
def pull_euro_equities_snapshot_safe(
    formation_date,
    max_retries: int = 3,
    sleep_s: float = 2.0,
) -> pd.DataFrame:
    """
    Safe wrapper: cache + retry + minimal field fallback.

    Returns standardized columns:
      RIC, name, hq_country, hq_code, trbc_sector_code, trbc_sector, mcap_eur

    Plus extra IDs + requested market fields:
      RIC_snapshot, RIC_current, ISIN, SEDOL, MasterID,
      FreeFloat, FreeFloatPct, Volume, PriceClose, InstrumentType
    """
    formation_date = pd.Timestamp(formation_date).normalize()
    date_iso = formation_date.strftime("%Y-%m-%d")

    # --- cache first ---
    p = _cache_path(formation_date)
    if p.exists():
        return pd.read_parquet(p)

    # --- dynamic euro country universe for this quarter ---
    universe = [make_screen_euro_all_for_quarter(formation_date)]

    # Strategy:
    # 1) try full fields incl CompanyMarketCap + MarketCap
    # 2) if backend 400/field issue, try only CompanyMarketCap
    # 3) if still, try only MarketCap
    base = [
        "TR.RIC","TR.PrimaryRIC","TR.ISIN","TR.SEDOL",
        "TR.CommonName","TR.HeadquartersCountry","TR.HQCountryCode",
        "TR.TRBCEconSectorCode","TR.TRBCEconomicSector",
        "TR.FreeFloat","TR.FreeFloatPct","TR.Volume","TR.PriceClose","TR.InstrumentType",
    ]

    field_sets = [
        base + ["TR.CompanyMarketCap","TR.MarketCap"],
        base + ["TR.CompanyMarketCap"],
        base + ["TR.MarketCap"],
    ]

    params = {"CURN":"EUR","RH":"In","CH":"Fd","SDate":date_iso,"EDate":date_iso}

    # IMPORTANT FIX #1: even the SCREEN string can sometimes generate empties downstream;
    # we still protect at the ld.get_data level via _safe_get_data().

    last_err = None

    for fields in field_sets:
        for attempt in range(1, max_retries + 1):
            try:
                ld.open_session()
                try:
                    df = _safe_get_data(
                        universe=universe,              # SCREEN string list
                        fields=fields,
                        parameters=params,
                        max_retries=max_retries,
                        base_sleep=sleep_s,
                    ).copy()
                finally:
                    ld.close_session()

                # If backend returns empty, still cache empty (prevents infinite retries)
                if df is None or len(df) == 0:
                    out = pd.DataFrame(columns=[
                        "RIC","name","hq_country","hq_code",
                        "trbc_sector_code","trbc_sector","mcap_eur",
                        "RIC_snapshot","RIC_current","ISIN","SEDOL","MasterID",
                        "FreeFloat","FreeFloatPct","Volume","PriceClose","InstrumentType",
                    ])
                    out.to_parquet(p, index=False)
                    return out

                # --- map columns robustly (TR.* OR friendly names) ---
                cols = list(df.columns)
                lower = {str(c).lower(): c for c in cols}

                def get_col(*names):
                    for n in names:
                        if n in df.columns:
                            return n
                        nl = str(n).lower()
                        if nl in lower:
                            return lower[nl]
                    return None

                col_ric = get_col("RIC","TR.RIC","Instrument")
                col_name = get_col("Company Common Name","TR.CommonName","TR.COMMONNAME")
                col_hq_country = get_col("Country of Headquarters","TR.HeadquartersCountry","TR.HEADQUARTERSCOUNTRY")
                col_hq_code = get_col("Country ISO Code of Headquarters","TR.HQCountryCode","TR.HQCOUNTRYCODE")
                col_trbc_code = get_col("TRBC Economic Sector Code","TR.TRBCEconSectorCode","TR.TRBCECONSECTORCODE")
                col_trbc_name = get_col("TRBC Economic Sector Name","TR.TRBCEconomicSector","TR.TRBCECONOMICSECTOR")
                col_mcap = get_col(
                    "Company Market Cap","TR.CompanyMarketCap","TR.MarketCap","Market Cap",
                    "TR.COMPANYMARKETCAP","TR.MARKETCAP"
                )

                col_primary = get_col("TR.PrimaryRIC","TR.PRIMARYRIC","Primary Issue RIC","Primary RIC")
                col_isin    = get_col("TR.ISIN","TR.ISINCode","ISIN","TR.ISINCODE")
                col_sedol   = get_col("TR.SEDOL","SEDOL","TR.SEDOLCode","TR.SEDOLCODE")

                col_ff      = get_col("TR.FreeFloat","Free Float","TR.FREEFLOAT")
                col_ffpct   = get_col("TR.FreeFloatPct","Free Float (Percent)","Free Float Percent","TR.FREEFLOATPCT")
                col_vol     = get_col("TR.Volume","Volume","TR.VOLUME")
                col_px      = get_col("TR.PriceClose","Price Close","Close Price","TR.PRICECLOSE","TR.CLOSEPRICE")
                col_type    = get_col("TR.InstrumentType","Instrument Type","TR.INSTRUMENTTYPE")

                if col_ric is None:
                    raise KeyError(f"No RIC column. Got: {cols}")
                if col_mcap is None:
                    raise KeyError(f"No MarketCap column. Got: {cols}")

                out = pd.DataFrame({
                    # standardized columns
                    "RIC": df[col_ric],
                    "name": df[col_name] if col_name else pd.NA,
                    "hq_country": df[col_hq_country] if col_hq_country else pd.NA,
                    "hq_code": df[col_hq_code] if col_hq_code else pd.NA,
                    "trbc_sector_code": df[col_trbc_code] if col_trbc_code else pd.NA,
                    "trbc_sector": df[col_trbc_name] if col_trbc_name else pd.NA,
                    "mcap_eur": df[col_mcap],

                    # IDs
                    "RIC_snapshot": df[col_ric],
                    "RIC_current": df[col_primary] if col_primary else df[col_ric],
                    "ISIN": df[col_isin] if col_isin else pd.NA,
                    "SEDOL": df[col_sedol] if col_sedol else pd.NA,

                    # requested market fields
                    "FreeFloat": df[col_ff] if col_ff else pd.NA,
                    "FreeFloatPct": df[col_ffpct] if col_ffpct else pd.NA,
                    "Volume": df[col_vol] if col_vol else pd.NA,
                    "PriceClose": df[col_px] if col_px else pd.NA,
                    "InstrumentType": df[col_type] if col_type else pd.NA,
                })

                # IMPORTANT FIX #2: clean empty strings in identifiers (prevents downstream bad merges)
                for idc in ["RIC", "RIC_snapshot", "RIC_current", "ISIN", "SEDOL"]:
                    out[idc] = out[idc].astype(str).str.strip()
                    out.loc[out[idc] == "", idc] = pd.NA

                # MasterID: prefer ISIN, fallback SEDOL, fallback RIC_current
                out["MasterID"] = out["ISIN"].fillna(out["SEDOL"]).fillna(out["RIC_current"])

                # numeric coercion
                for c in ["mcap_eur","FreeFloat","FreeFloatPct","Volume","PriceClose"]:
                    out[c] = pd.to_numeric(out[c], errors="coerce")

                # Drop rows without RIC (post-clean)
                out = out.dropna(subset=["RIC"]).copy()

                # cache (also good for reproducibility)
                out.to_parquet(p, index=False)
                return out

            except Exception as e:
                last_err = e
                time.sleep(sleep_s * attempt)

    raise last_err

### 2.2 Loop über Screener um für jeder Quartal alle Equities zu ziehen

In [28]:
def build_quarterly_euro_panel_safe(
    q_ends: pd.DatetimeIndex
) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = []
    failures = []

    for dt in q_ends:
        formation_date = pd.Timestamp(dt).normalize()
        date_iso = formation_date.strftime("%Y-%m-%d")
        eligibility_date = (formation_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

        print(f"Quarter (formation): {formation_date.date()} -> {date_iso} | eligibility: {eligibility_date}")

        try:
            snap = pull_euro_equities_snapshot_safe(formation_date).copy()

            # ensure date column exists and is the first column
            snap.insert(0, "date", formation_date)

            # ----------------------------------------------------
            # NEW: Hard-filter Ordinary Shares (extra safety layer)
            # ----------------------------------------------------
            if "InstrumentType" in snap.columns:
                snap["InstrumentType"] = snap["InstrumentType"].astype(str).str.strip()
                snap = snap[snap["InstrumentType"].str.lower() == "ordinary shares"].copy()

            # ----------------------------------------------------
            # Dedupe within quarter (MasterID if usable else RIC)
            # ----------------------------------------------------
            if "MasterID" in snap.columns:
                # Only use MasterID where present; otherwise fallback to RIC
                snap["_dedupe_key"] = snap["MasterID"].where(snap["MasterID"].notna(), snap["RIC"])
                snap = snap.drop_duplicates(subset=["date", "_dedupe_key"], keep="first").drop(columns=["_dedupe_key"])
            else:
                snap = snap.drop_duplicates(subset=["date", "RIC"], keep="first")

            # ----------------------------------------------------
            # Diagnostics (optional)
            # ----------------------------------------------------
            if "MasterID" in snap.columns:
                n_miss = int(snap["MasterID"].isna().sum())
                if n_miss > 0:
                    print(f"  ⚠️ missing MasterID: {n_miss} / {len(snap)}")

            if "InstrumentType" in snap.columns:
                top_types = snap["InstrumentType"].value_counts(dropna=False).head(3).to_dict()
                print(f"  InstrumentType top3: {top_types}")

            out.append(snap)

        except Exception as e:
            failures.append({
                "date": date_iso,
                "formation_date": date_iso,
                "eligibility_date": eligibility_date,
                "error": str(e)[:500],
            })
            print("  ❌ failed:", str(e)[:200], "...")

    panel = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    fail_df = pd.DataFrame(failures)

    return panel, fail_df


panel_all, fail_log = build_quarterly_euro_panel_safe(q_ends)

print("Done. Panel rows:", len(panel_all))
print("Failures:", len(fail_log))

Quarter (formation): 1998-12-31 -> 1998-12-31 | eligibility: 1999-01-01
  InstrumentType top3: {'Ordinary Shares': 2782}
Quarter (formation): 1999-03-31 -> 1999-03-31 | eligibility: 1999-04-01
  InstrumentType top3: {'Ordinary Shares': 2809}
Quarter (formation): 1999-06-30 -> 1999-06-30 | eligibility: 1999-07-01
  InstrumentType top3: {'Ordinary Shares': 2810}
Quarter (formation): 1999-09-30 -> 1999-09-30 | eligibility: 1999-10-01
  InstrumentType top3: {'Ordinary Shares': 2810}
Quarter (formation): 1999-12-31 -> 1999-12-31 | eligibility: 2000-01-01
  InstrumentType top3: {'Ordinary Shares': 2809}
Quarter (formation): 2000-03-31 -> 2000-03-31 | eligibility: 2000-04-01
  InstrumentType top3: {'Ordinary Shares': 2810}
Quarter (formation): 2000-06-30 -> 2000-06-30 | eligibility: 2000-07-01
  InstrumentType top3: {'Ordinary Shares': 2810}
Quarter (formation): 2000-09-30 -> 2000-09-30 | eligibility: 2000-10-01
  InstrumentType top3: {'Ordinary Shares': 2810}
Quarter (formation): 2000-12-31 

In [29]:
panel_all.to_parquet(DATA_DIR / "euro_equities_all_quarterly_snapshots.parquet", index=False)
fail_log.to_csv(DATA_DIR / "euro_equities_pull_failures.csv", index=False)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time

# -----------------------------------------
# Config
# -----------------------------------------
CACHE_DIR = DATA_DIR / "euro_snap_cache"
assert CACHE_DIR.exists(), f"Cache dir not found: {CACHE_DIR}"

# We only want to add these two (optionally with fallbacks)
FF_PCT_FIELDSETS = [
    ["TR.FreeFloatPct"],           # primary
    ["TR.IssuerFreeFloatPct"],     # fallback (often available)
]
PRICE_FIELDSETS = [
    ["TR.PriceClose"],             # primary
    ["TR.ClosePrice"],             # fallback
]

def _clean_universe(rics):
    s = pd.Series(rics, dtype="object")
    s = s.dropna().astype(str).str.strip()
    s = s[s != ""]
    # unique preserving order
    return s.drop_duplicates().tolist()

def _safe_get_data(universe, fields, parameters, max_retries=5, base_sleep=0.8):
    last = None
    for r in range(max_retries):
        try:
            u = _clean_universe(universe)
            if not u:
                return None
            df = ld.get_data(universe=u, fields=fields, parameters=parameters)
            if df is not None and len(df) > 0:
                return df
            return df
        except Exception as e:
            last = e
            time.sleep(base_sleep * (2**r) + 0.2*np.random.random())
    raise last

def _best_col(df, candidates):
    cols = list(df.columns)
    lower = {str(c).lower(): c for c in cols}
    for c in candidates:
        if c in df.columns:
            return c
        cl = str(c).lower()
        if cl in lower:
            return lower[cl]
    return None

def _standardize_return(df_raw):
    """
    Map LSEG return to a simple frame: RIC + requested fields.
    Works with TR.* headers or friendly headers.
    """
    if df_raw is None or len(df_raw) == 0:
        return pd.DataFrame()

    ric_col = _best_col(df_raw, ["Instrument", "RIC", "TR.RIC"])
    if ric_col is None:
        # Sometimes first column is instrument id
        ric_col = df_raw.columns[0]

    out = pd.DataFrame({"RIC": df_raw[ric_col].astype(str).str.strip()})
    out = out[out["RIC"] != ""].copy()

    # FreeFloatPct
    ff_col = _best_col(
        df_raw,
        ["TR.FreeFloatPct", "Free Float Percent", "Free Float Pct", "TR.ISSUERFREEFLOATPCT", "TR.IssuerFreeFloatPct"]
    )
    if ff_col is not None:
        out["FreeFloatPct"] = pd.to_numeric(df_raw[ff_col], errors="coerce")

    # PriceClose
    px_col = _best_col(
        df_raw,
        ["TR.PriceClose", "Close Price", "TR.ClosePrice", "TR.CLOSEPRICE", "TR.PRICECLOSE"]
    )
    if px_col is not None:
        out["PriceClose"] = pd.to_numeric(df_raw[px_col], errors="coerce")

    return out.drop_duplicates("RIC")

def augment_snapshot_file(path: Path, batch_size=200, max_retries=5):
    df = pd.read_parquet(path)
    if df is None or df.empty:
        return False

    # If columns don’t exist, create them
    if "FreeFloatPct" not in df.columns:
        df["FreeFloatPct"] = pd.NA
    if "PriceClose" not in df.columns:
        df["PriceClose"] = pd.NA

    # Determine if we need to fetch (missing column values)
    need_ff = df["FreeFloatPct"].isna().all() or df["FreeFloatPct"].isna().mean() > 0.95
    need_px = df["PriceClose"].isna().all() or df["PriceClose"].isna().mean() > 0.95
    if not (need_ff or need_px):
        return False

    # Use snapshot date from filename: euro_snapshot_YYYY-MM-DD.parquet
    date_str = path.stem.replace("euro_snapshot_", "")
    try:
        asof = pd.to_datetime(date_str).strftime("%Y-%m-%d")
    except Exception:
        # if weird filename, skip
        return False

    # Pick universe column
    ric_col = "RIC"
    if "RIC_snapshot" in df.columns:
        ric_col = "RIC_snapshot"

    universe = _clean_universe(df[ric_col].tolist())
    if not universe:
        return False

    params = {"CURN": "EUR", "RH": "In", "CH": "Fd", "SDate": asof, "EDate": asof}

    # Build fields to request (try minimal)
    # We do it in two passes (ff then px) to avoid 400 due to bad field combos.
    updates = []

    # --- FreeFloatPct ---
    if need_ff:
        ff_done = False
        for fs in FF_PCT_FIELDSETS:
            try:
                # request also RIC so we can merge robustly
                raw = _safe_get_data(universe, fields=["TR.RIC"] + fs, parameters=params, max_retries=max_retries)
                part = _standardize_return(raw)
                if not part.empty and "FreeFloatPct" in part.columns:
                    updates.append(part[["RIC", "FreeFloatPct"]])
                    ff_done = True
                    break
            except Exception:
                continue
        if not ff_done:
            # keep NA; don’t fail whole file
            pass

    # --- PriceClose ---
    if need_px:
        px_done = False
        for fs in PRICE_FIELDSETS:
            try:
                raw = _safe_get_data(universe, fields=["TR.RIC"] + fs, parameters=params, max_retries=max_retries)
                part = _standardize_return(raw)
                if not part.empty and "PriceClose" in part.columns:
                    updates.append(part[["RIC", "PriceClose"]])
                    px_done = True
                    break
            except Exception:
                continue
        if not px_done:
            pass

    if not updates:
        return False

    upd = updates[0]
    for u in updates[1:]:
        upd = upd.merge(u, on="RIC", how="outer")

    # Merge back into snapshot
    df_merge_key = df[ric_col].astype(str).str.strip()
    df = df.copy()
    df["_merge_key"] = df_merge_key

    upd = upd.copy()
    upd["_merge_key"] = upd["RIC"].astype(str).str.strip()

    df = df.merge(upd.drop(columns=["RIC"]), on="_merge_key", how="left", suffixes=("", "_new"))

    # Fill only missing
    if "FreeFloatPct_new" in df.columns:
        df["FreeFloatPct"] = pd.to_numeric(df["FreeFloatPct"], errors="coerce")
        df["FreeFloatPct_new"] = pd.to_numeric(df["FreeFloatPct_new"], errors="coerce")
        df["FreeFloatPct"] = df["FreeFloatPct"].fillna(df["FreeFloatPct_new"])
        df = df.drop(columns=["FreeFloatPct_new"])

    if "PriceClose_new" in df.columns:
        df["PriceClose"] = pd.to_numeric(df["PriceClose"], errors="coerce")
        df["PriceClose_new"] = pd.to_numeric(df["PriceClose_new"], errors="coerce")
        df["PriceClose"] = df["PriceClose"].fillna(df["PriceClose_new"])
        df = df.drop(columns=["PriceClose_new"])

    df = df.drop(columns=["_merge_key"])

    # Write back atomically
    tmp = path.with_suffix(".parquet.tmp")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)

    return True

def augment_all_cached_snapshots(limit=None, batch_size=200):
    files = sorted(CACHE_DIR.glob("euro_snapshot_*.parquet"))
    if limit:
        files = files[:limit]

    print(f"Found {len(files)} snapshot files in cache.")
    changed = 0

    ld.open_session()
    try:
        for i, p in enumerate(files, start=1):
            ok = augment_snapshot_file(p, batch_size=batch_size)
            if ok:
                changed += 1
                print(f"[{i}/{len(files)}] updated: {p.name}")
            else:
                if i % 25 == 0:
                    print(f"[{i}/{len(files)}] ...")
    finally:
        ld.close_session()

    print(f"Done. Updated {changed}/{len(files)} files.")

# ---------------------------
# RUN
# ---------------------------
augment_all_cached_snapshots()

Found 109 snapshot files in cache.
[1/109] updated: euro_snapshot_1998-12-31.parquet
[2/109] updated: euro_snapshot_1999-03-31.parquet
[3/109] updated: euro_snapshot_1999-06-30.parquet
[4/109] updated: euro_snapshot_1999-09-30.parquet
[5/109] updated: euro_snapshot_1999-12-31.parquet
[6/109] updated: euro_snapshot_2000-03-31.parquet
[7/109] updated: euro_snapshot_2000-06-30.parquet
[8/109] updated: euro_snapshot_2000-09-30.parquet
[9/109] updated: euro_snapshot_2000-12-31.parquet
[10/109] updated: euro_snapshot_2001-03-31.parquet
[11/109] updated: euro_snapshot_2001-06-30.parquet
[12/109] updated: euro_snapshot_2001-09-30.parquet
[13/109] updated: euro_snapshot_2001-12-31.parquet
[14/109] updated: euro_snapshot_2002-03-31.parquet
[15/109] updated: euro_snapshot_2002-06-30.parquet
[16/109] updated: euro_snapshot_2002-09-30.parquet
[17/109] updated: euro_snapshot_2002-12-31.parquet
[18/109] updated: euro_snapshot_2003-03-31.parquet
[19/109] updated: euro_snapshot_2003-06-30.parquet
[20/1

/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1060:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1084:RuntimeWarning: invalid value encountered in cast


[55/109] updated: euro_snapshot_2012-06-30.parquet


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1060:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1084:RuntimeWarning: invalid value encountered in cast


[56/109] updated: euro_snapshot_2012-09-30.parquet


...

[61/109] updated: euro_snapshot_2013-12-31.parquet
[62/109] updated: euro_snapshot_2014-03-31.parquet
[63/109] updated: euro_snapshot_2014-06-30.parquet
[64/109] updated: euro_snapshot_2014-09-30.parquet


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1060:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1084:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1060:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1084:RuntimeWarning: invalid value encountered in cast


[65/109] updated: euro_snapshot_2014-12-31.parquet
[66/109] updated: euro_snapshot_2015-03-31.parquet
[67/109] updated: euro_snapshot_2015-06-30.parquet
[68/109] updated: euro_snapshot_2015-09-30.parquet
[69/109] updated: euro_snapshot_2015-12-31.parquet


/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1060:RuntimeWarning: invalid value encountered in cast
/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Code/.venv Masterarbeit/lib/python3.13/site-packages/pandas/core/dtypes/cast.py:1084:RuntimeWarning: invalid value encountered in cast


[70/109] updated: euro_snapshot_2016-03-31.parquet
[71/109] updated: euro_snapshot_2016-06-30.parquet
[72/109] updated: euro_snapshot_2016-09-30.parquet
[73/109] updated: euro_snapshot_2016-12-31.parquet
[74/109] updated: euro_snapshot_2017-03-31.parquet
[75/109] updated: euro_snapshot_2017-06-30.parquet
[76/109] updated: euro_snapshot_2017-09-30.parquet
[77/109] updated: euro_snapshot_2017-12-31.parquet
[78/109] updated: euro_snapshot_2018-03-31.parquet
[79/109] updated: euro_snapshot_2018-06-30.parquet
[80/109] updated: euro_snapshot_2018-09-30.parquet
[81/109] updated: euro_snapshot_2018-12-31.parquet
[82/109] updated: euro_snapshot_2019-03-31.parquet
[83/109] updated: euro_snapshot_2019-06-30.parquet
[84/109] updated: euro_snapshot_2019-09-30.parquet
[85/109] updated: euro_snapshot_2019-12-31.parquet
[86/109] updated: euro_snapshot_2020-03-31.parquet
[87/109] updated: euro_snapshot_2020-06-30.parquet
[88/109] updated: euro_snapshot_2020-09-30.parquet


### 2.3 Top 500 in jedem Quartal behalten

In [30]:
# ---------------------------------------------
# Exclude: (1) NOT Ordinary Shares (keep only)
#          (2) Financials (string-based, robust)
# ---------------------------------------------
panel_nonfin = panel_all.copy()

# 1) Keep only Ordinary Shares
panel_nonfin = panel_nonfin[
    panel_nonfin["InstrumentType"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "ordinary shares"
].copy()

# 2) Exclude Financials (keep logic as before)
panel_nonfin = panel_nonfin[
    panel_nonfin["trbc_sector"]
    .astype(str)
    .str.strip()
    .str.lower()
    != "financials"
].copy()

# 3) Sort + rank by market cap within each quarter
def top500_unique_with_rank(g):
    # 1) Sort nach Market Cap (größte zuerst)
    g = g.sort_values(["mcap_eur", "RIC"], ascending=[False, True]).copy()
    
    # 2) Doppelte RICs innerhalb Quartal entfernen
    g = g.drop_duplicates(subset=["RIC"], keep="first")
    
    # 3) Rank neu berechnen
    g["rank_mcap"] = range(1, len(g) + 1)
    
    # 4) Top 500 nehmen
    return g.head(500)

euro500 = (
    panel_nonfin
    .groupby("date", group_keys=False)
    .apply(top500_unique_with_rank)
    .reset_index(drop=True)
)

save_parquet(euro500, "euro500")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500.parquet


/var/folders/r2/vtz74sz14fx185wt9m3c781w0000gn/T/ipykernel_76297/2464159880.py:42:FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [31]:
euro500.groupby("date")["RIC"].nunique().describe()

count    109.0
mean     500.0
std        0.0
min      500.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: RIC, dtype: float64

In [32]:
euro500["RIC"].nunique()

1495

In [33]:
def assert_euro500_hq_countries_ok(euro500: pd.DataFrame) -> None:
    """
    Prints 'OK ✅' if, for every quarter, all HQ countries present in euro500
    are euro-eligible for the subsequent return period (per euro_hq_codes_for_quarter).
    Otherwise prints a short FAIL message.
    """
    for dt, g in euro500.groupby("date"):
        formation_date = pd.Timestamp(dt).normalize()
        allowed = set(euro_hq_codes_for_quarter(formation_date))

        present = set(
            g["hq_code"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
            .unique()
        )

        viol = sorted(present - allowed)

        if len(viol) > 0:
            print("FAIL ❌")
            print("First problematic quarter (formation):", formation_date.date())
            print("Countries present but not euro-eligible:", viol)
            return

    print("OK ✅ All HQ countries in euro500 are correctly euro-eligible in every quarter.")

assert_euro500_hq_countries_ok(euro500)

OK ✅ All HQ countries in euro500 are correctly euro-eligible in every quarter.


## Analysis

In [34]:
country_share = (
    euro500
    .groupby(["date", "hq_code"])["RIC"]
    .nunique()
    .groupby("hq_code")
    .mean()
    .sort_values(ascending=False)
)

country_share_pct = 100 * country_share / country_share.sum()

country_share_pct.head(15)

hq_code
FR    23.159500
DE    20.846433
IT     9.311719
ES     8.355146
NL     7.079717
BE     6.290679
FI     5.831309
IE     4.813487
AT     4.119927
GR     3.145625
LU     2.385126
PT     2.287847
HR     0.528658
SI     0.436069
SK     0.319083
Name: RIC, dtype: float64

In [35]:
sector_share = (
    euro500
    .groupby(["date", "trbc_sector"])["RIC"]
    .nunique()
    .groupby("trbc_sector")
    .mean()
    .sort_values(ascending=False)
)

sector_share_pct = 100 * sector_share / sector_share.sum()
sector_share_pct

trbc_sector
Industrials                                   21.397402
Consumer Cyclicals                            18.361519
Technology                                    12.989778
Basic Materials                               11.531238
Healthcare                                     8.999813
Consumer Non-Cyclicals                         8.438695
Real Estate                                    7.214107
Utilities                                      5.888992
Energy                                         4.730202
Academic & Educational Services                0.249030
Institutions, Associations & Organizations     0.199224
Name: RIC, dtype: float64

In [36]:
# Prüfe leere Strings
empty_string_count = (euro500["ISIN"] == "").sum()
print("Empty string ISINs:", empty_string_count)

# Prüfe Whitespaces
whitespace_count = (euro500["ISIN"].astype(str).str.strip() == "").sum()
print("Whitespace-only ISINs:", whitespace_count)

Empty string ISINs: 0
Whitespace-only ISINs: 0


## Kopie für Dashboard erstellen

In [37]:
# --- gewünschte Spalten auswählen ---
export_df = euro500[[
    "date",
    "name",
    "hq_country",
    "trbc_sector",
    "mcap_eur",
    "ISIN",
    "rank_mcap",
    "MasterID"
]].copy()

# Umbenennen
export_df = export_df.rename(columns={
    "trbc_sector": "sector",
    "ISIN": "isin"
})

# --- mcap: letzte 6 Ziffern auf 0 setzen ---
# Beispiel: 1234567890 -> 1234000000
export_df["mcap_eur"] = (
    (export_df["mcap_eur"] // 1_000_000) * 1_000_000
).astype("Int64")

# --- Speicherpfad auf deinem Mac ---
output_path = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/EURO500 Dashboard/euro500_dashboard_table.parquet"
)

# Ordner sicherstellen
output_path.parent.mkdir(parents=True, exist_ok=True)

# Speichern
export_df.to_parquet(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/EURO500 Dashboard/euro500_dashboard_table.parquet
